# ProteinForge — ensayo de rediseño / sequence redesign experiment

**Objetivo:** estructura experimental conocida → secuencias ProteinMPNN → predicción ESMFold de la primera muestra → comparación y ZIP verificable.

**Goal:** known experimental backbone → ProteinMPNN sequences → ESMFold prediction of the first sample → comparison and verifiable ZIP.

Este caso usa ubiquitina pública **1UBQ, cadena A, 76 residuos**. Rediseña una secuencia; **no genera una forma nueva ni demuestra función o estabilidad experimental**. Cada ejecución conserva sus propias validaciones y mediciones. / This public 76-residue case redesigns a sequence, does not generate a new backbone, and does not establish biological function or experimental stability. Each run retains its own validation and measurements.

En **Entorno de ejecución → Cambiar tipo de entorno → GPU**, selecciona una GPU gratuita, si está disponible. Después usa **Ejecutar todas**. No contrates recursos. Colab no garantiza GPU ni duración. / Select a free GPU runtime, if available, then Run all. Do not purchase resources; availability and session duration are not guaranteed.

El cálculo se realiza en Colab. No se monta Drive, no se usa Supabase, no se solicita una clave y no se modifica ProteinForge publicado. Los pesos de predicción requieren descargar unos **8,44 GB** en la máquina de Colab. / Computation runs in Colab. No Drive mount, Supabase access, API key or production change is used. Prediction weights download approximately 8.44 GB into the Colab machine.

Versiones y hashes de entradas fijados; Torch/CUDA/GPU del entorno se registran. Repetición de generación con la misma semilla comprobada; igualdad bit a bit entre equipos distintos no garantizada. / Input versions and hashes are pinned; runtime Torch/CUDA/GPU are recorded. Generation is repeated with the same seed; bitwise equality across different devices is not guaranteed.

Fuentes / Sources: [ProteinMPNN](https://github.com/dauparas/ProteinMPNN), [ESMFold](https://huggingface.co/facebook/esmfold_v1), [Transformers ESM](https://huggingface.co/docs/transformers/en/model_doc/esm), [1UBQ](https://www.rcsb.org/structure/1UBQ), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
#@title Preparar el entorno de Colab / Prepare the Colab environment
PROGRAM_TEXT = '"""Interactive Colab experiment. No web API, external inference or private input."""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import OrderedDict\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport urllib.request\nimport zipfile\n\nfrom constraints import constraint_report, prepare_constraints, verify_constraints\n\n\nRESIDUES = dict(zip(\n    "ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split(),\n    "ARNDCQEGHILKMFPSTWYV",\n))\nBACKBONE = {"N", "CA", "C", "O"}\n\n\ndef read_json(path: Path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef write_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    text = json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(text, encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path: Path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download(url: str, path: Path, expected: str, config: dict):\n    if path.exists() and sha256(path) == expected:\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".part")\n    request = urllib.request.Request(url, headers={"User-Agent": "ProteinForge-experiment"})\n    with urllib.request.urlopen(request, timeout=config["timeoutSeconds"]) as response:\n        with temporary.open("wb") as stream:\n            shutil.copyfileobj(response, stream, length=config["chunkBytes"])\n    if sha256(temporary) != expected:\n        raise ValueError(f"Downloaded file hash mismatch: {path.name}")\n    temporary.replace(path)\n\n\ndef parse_pdb(text: str, chain: str, expected_sequence: str | None = None):\n    """Read one standard polymer chain in the first model, with explicit residue keys."""\n    residues = OrderedDict()\n    model_seen = False\n    atom_count = 0\n    for line in text.splitlines():\n        if line.startswith("MODEL "):\n            if model_seen:\n                break\n            model_seen = True\n        if line.startswith("ENDMDL"):\n            break\n        if not line.startswith("ATOM  ") or line[21:22] != chain:\n            continue\n        if len(line) < 60:\n            raise ValueError("Truncated PDB atom")\n        if line[16] not in (" ", "A"):\n            continue\n        residue_name = line[17:20].strip()\n        if residue_name not in RESIDUES:\n            raise ValueError("Non-standard polymer residue")\n        coordinates = [float(line[a:b]) for a, b in ((30, 38), (38, 46), (46, 54))]\n        occupancy = float(line[54:60])\n        if not all(math.isfinite(value) for value in (*coordinates, occupancy)):\n            raise ValueError("Non-finite atom data")\n        if occupancy <= 0:\n            continue\n        key = (int(line[22:26]), line[26])\n        residue = residues.setdefault(key, {"name": residue_name, "atoms": {}, "lines": []})\n        if residue["name"] != residue_name:\n            raise ValueError("Ambiguous residue identity")\n        atom = line[12:16].strip()\n        if atom in residue["atoms"]:\n            raise ValueError("Duplicate atom in residue")\n        residue["atoms"][atom] = coordinates\n        residue["lines"].append(line)\n        atom_count += 1\n    if not residues:\n        raise ValueError("Polymer chain is absent")\n    for residue in residues.values():\n        if not BACKBONE <= residue["atoms"].keys():\n            raise ValueError("Incomplete backbone")\n    keys = list(residues)\n    if keys != sorted(keys):\n        raise ValueError("Unordered residue numbering")\n    sequence = "".join(RESIDUES[residue["name"]] for residue in residues.values())\n    if expected_sequence is not None and sequence != expected_sequence:\n        raise ValueError("PDB sequence does not match the exact expected sequence")\n    return {\n        "chain": chain,\n        "sequence": sequence,\n        "residueKeys": [[number, insertion.strip()] for number, insertion in residues],\n        "ca": [residue["atoms"]["CA"] for residue in residues.values()],\n        "atomCount": atom_count,\n        "pdb": "\\n".join(line for residue in residues.values() for line in residue["lines"]) + "\\nTER\\nEND\\n",\n    }\n\n\ndef parse_fasta(text: str):\n    entries = []\n    for line in text.splitlines():\n        line = line.strip()\n        if not line:\n            continue\n        if line.startswith(">"):\n            entries.append({"header": line[1:], "sequence": ""})\n        elif entries:\n            entries[-1]["sequence"] += line\n        else:\n            raise ValueError("FASTA sequence before header")\n    if not entries or any(not entry["sequence"] for entry in entries):\n        raise ValueError("Empty FASTA entry")\n    for entry in entries:\n        if any(aa not in RESIDUES.values() for aa in entry["sequence"]):\n            raise ValueError("Unsupported candidate alphabet or multiple chains")\n    return entries\n\n\ndef compare_ca(reference, prediction):\n    """Least-squares rigid superposition without reflection; coordinates in angstroms."""\n    import numpy as np\n\n    first = np.asarray(reference, dtype=np.float64)\n    second = np.asarray(prediction, dtype=np.float64)\n    if first.shape != second.shape or first.ndim != 2 or first.shape[1] != 3 or len(first) < 3:\n        raise ValueError("Incompatible CA coordinates")\n    if not np.isfinite(first).all() or not np.isfinite(second).all():\n        raise ValueError("Non-finite CA coordinates")\n    first_center = first.mean(axis=0)\n    second_center = second.mean(axis=0)\n    x, y = first - first_center, second - second_center\n    u, _, vt = np.linalg.svd(y.T @ x)\n    correction = np.eye(3)\n    correction[-1, -1] = 1 if np.linalg.det(u @ vt) >= 0 else -1\n    rotation = u @ correction @ vt\n    aligned = y @ rotation + first_center\n    distances = np.linalg.norm(aligned - first, axis=1)\n    return {\n        "method": "Kabsch CA least-squares, proper rotation, no outlier exclusion",\n        "residuesCompared": len(first),\n        "caRmsdAngstrom": float(np.sqrt(np.mean(distances ** 2))),\n        "caDistancesAngstrom": distances.tolist(),\n        "rotationRowVectors": rotation.tolist(),\n        "translation": (first_center - second_center @ rotation).tolist(),\n    }\n\n\ndef gpu_memory_mib():\n    result = subprocess.run(\n        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],\n        capture_output=True, text=True, timeout=15, check=True,\n    )\n    return max(float(line.strip()) for line in result.stdout.splitlines() if line.strip())\n\n\n@contextmanager\ndef measurement(root: Path, name: str, interval: float):\n    import psutil\n\n    stop = threading.Event()\n    process = psutil.Process()\n    metrics = {"startedAt": datetime.now(timezone.utc).isoformat(), "sampleSeconds": interval,\n               "peakProcessTreeRssBytes": 0, "peakGpuDeviceUsedMiB": None,\n               "samplerErrors": [], "status": "running"}\n\n    def sample():\n        while not stop.is_set():\n            try:\n                rss = sum(p.memory_info().rss for p in [process, *process.children(recursive=True)] if p.is_running())\n                metrics["peakProcessTreeRssBytes"] = max(metrics["peakProcessTreeRssBytes"], rss)\n                used = gpu_memory_mib()\n                metrics["peakGpuDeviceUsedMiB"] = max(metrics["peakGpuDeviceUsedMiB"] or 0, used)\n            except (OSError, subprocess.SubprocessError, psutil.Error, ValueError) as error:\n                if not metrics["samplerErrors"]:\n                    metrics["samplerErrors"].append(type(error).__name__)\n            stop.wait(interval)\n\n    started = time.perf_counter()\n    thread = threading.Thread(target=sample, daemon=True)\n    thread.start()\n    try:\n        yield metrics\n        metrics["status"] = "completed"\n    except BaseException as error:\n        metrics.update(status="failed", errorType=type(error).__name__, error=str(error))\n        raise\n    finally:\n        stop.set()\n        thread.join(timeout=20)\n        metrics["seconds"] = time.perf_counter() - started\n        metrics["finishedAt"] = datetime.now(timezone.utc).isoformat()\n        write_json(root / "measurements" / f"{name}.json", metrics)\n\n\ndef environment(root: Path, config: dict):\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A managed Colab GPU runtime is required; switch the hardware accelerator to GPU")\n    gpu = torch.cuda.get_device_properties(0)\n    if gpu.total_memory < config["prediction"]["minimumGpuGiB"] * 1024 ** 3:\n        raise RuntimeError("The allocated GPU has insufficient VRAM for this experiment")\n    if shutil.disk_usage(root).free < config["prediction"]["minimumFreeDiskGiB"] * 1024 ** 3:\n        raise RuntimeError("Insufficient free runtime disk space")\n    versions = {}\n    for package in ("torch", "numpy", "transformers", "accelerate", "huggingface-hub", "tokenizers", "psutil", "safetensors"):\n        versions[package] = importlib.metadata.version(package)\n    write_json(root / "environment.json", {\n        "python": sys.version, "platform": platform.platform(), "packages": versions,\n        "torchCuda": torch.version.cuda, "gpu": gpu.name, "gpuMemoryBytes": gpu.total_memory,\n        "freeDiskBytes": shutil.disk_usage(root).free,\n        "experimentScriptSha256": sha256(Path(__file__)),\n        "configSha256": sha256(root / "config.json"),\n    })\n    (root / "pip-freeze.txt").write_text(subprocess.check_output(\n        [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")\n\n\ndef prepare(root: Path, config: dict):\n    environment(root, config)\n    source = root / "downloads" / "reference.pdb"\n    reference_config = config["reference"]\n    download(reference_config["url"], source, reference_config["sha256"], config["download"])\n    reference = parse_pdb(source.read_text(), reference_config["chain"], reference_config["sequence"])\n    prepare_constraints(root, config, reference)\n    (root / "reference.pdb").write_text(reference["pdb"], encoding="ascii")\n    (root / "reference.fasta").write_text(f">{reference_config[\'id\']} chain {reference_config[\'chain\']} experimental reference\\n{reference[\'sequence\']}\\n")\n    write_json(root / "reference.json", {key: value for key, value in reference.items() if key != "pdb"})\n    for file, digest in config["design"]["files"].items():\n        download(config["design"]["rawBase"] + file, root / "downloads" / "ProteinMPNN" / file,\n                 digest, config["download"])\n    write_json(root / "provenance.json", {\n        "schemaVersion": config["schemaVersion"], "experiment": "known-backbone-sequence-redesign",\n        "reference": reference_config, "design": config["design"], "prediction": config["prediction"],\n        "limitations": config["limitations"],\n    })\n    print("Public reference and pinned ProteinMPNN files verified", flush=True)\n\n\ndef design(root: Path, config: dict):\n    design_config = config["design"]\n    source = root / "downloads" / "ProteinMPNN"\n    reference = read_json(root / "reference.json")\n    if design_config.get("constraints") is not None:\n        write_json(root / "constraints-validation.json", {"status": "incomplete"})\n    complete_reference = parse_pdb((root / "reference.pdb").read_text(), config["reference"]["chain"], reference["sequence"])\n    draft = prepare_constraints(root, config, complete_reference)\n    outputs = []\n    for run_name in ("design", "design-repeat"):\n        command = [sys.executable, str(source / "protein_mpnn_run.py"),\n                   "--path_to_model_weights", str(source / "vanilla_model_weights"),\n                   "--model_name", design_config["model"], "--pdb_path", str(root / "reference.pdb"),\n                   "--pdb_path_chains", config["reference"]["chain"],\n                   "--out_folder", str(root / run_name), "--seed", str(design_config["seed"]),\n                   "--num_seq_per_target", str(design_config["samples"]), "--batch_size", "1",\n                   "--sampling_temp", str(design_config["temperature"]),\n                   "--max_length", str(design_config["maxLength"]), "--save_score", "1"]\n        if draft is not None:\n            start = command.index("--pdb_path")\n            del command[start:start + 4]\n            command.extend(["--jsonl_path", str(root / "design-input.jsonl"),\n                            "--fixed_positions_jsonl", str(root / "fixed-positions.jsonl")])\n        write_json(root / "commands" / f"{run_name}.json", command)\n        with measurement(root, run_name, config["measurement"]["sampleSeconds"]):\n            result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n            (root / f"{run_name}.log").write_text(result.stdout, encoding="utf-8")\n            print(result.stdout, flush=True)\n            result.check_returncode()\n        entries = parse_fasta((root / run_name / "seqs" / "reference.fa").read_text())\n        if entries[0]["sequence"] != (draft["sequence"] if draft else reference["sequence"]):\n            raise ValueError("ProteinMPNN reference sequence mismatch")\n        candidates = entries[1:]\n        if len(candidates) != design_config["samples"]:\n            raise ValueError("Unexpected number of generated samples")\n        if any(len(item["sequence"]) != len(reference["sequence"]) for item in candidates):\n            raise ValueError("Candidate length mismatch")\n        if draft:\n            verify_constraints(draft, candidates)\n        outputs.append(candidates)\n    if [item["sequence"] for item in outputs[0]] != [item["sequence"] for item in outputs[1]]:\n        raise RuntimeError("Repeated ProteinMPNN generation did not reproduce the sequences")\n    index = design_config["selectedSample"] - 1\n    candidate = outputs[0][index]\n    candidate.update(sample=index + 1, selectedBeforePrediction=True,\n                     sequenceSha256=hashlib.sha256(candidate["sequence"].encode("ascii")).hexdigest(),\n                     nativeSequenceRecovery=sum(a == b for a, b in zip(candidate["sequence"], reference["sequence"])) / len(reference["sequence"]))\n    write_json(root / "candidate.json", candidate)\n    (root / "candidate.fasta").write_text(f">ProteinForge sample {index + 1}; generated, not experimentally validated\\n{candidate[\'sequence\']}\\n")\n    write_json(root / "design-summary.json", {"candidates": outputs[0], "repeatSequencesEqual": True})\n    if draft:\n        write_json(root / "constraints-validation.json", constraint_report(draft, config))\n    print("Candidate selected by fixed sample index; repeat sequence check passed", flush=True)\n\n\ndef predict(root: Path, config: dict):\n    import numpy as np\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import EsmConfig, EsmForProteinFolding\n    from transformers.models.esm.openfold_utils import residue_constants\n\n    prediction = config["prediction"]\n    candidate = read_json(root / "candidate.json")\n    sequence = candidate["sequence"]\n    reference = parse_pdb((root / "reference.pdb").read_text(), config["reference"]["chain"], config["reference"]["sequence"])\n    draft = prepare_constraints(root, config, reference)\n    if draft:\n        verify_constraints(draft, [candidate])\n    torch.manual_seed(prediction["seed"])\n    torch.cuda.manual_seed_all(prediction["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.cuda.reset_peak_memory_stats()\n    with measurement(root, "model-download", config["measurement"]["sampleSeconds"]):\n        model_path = Path(snapshot_download(\n            repo_id=prediction["model"], revision=prediction["revision"],\n            allow_patterns=["config.json", prediction["weights"]], token=False,\n        ))\n        weights = model_path / prediction["weights"]\n        if weights.stat().st_size != prediction["weightsBytes"] or sha256(weights) != prediction["weightsSha256"]:\n            raise ValueError("Prediction weights do not match the pinned artifact")\n    with measurement(root, "model-load", config["measurement"]["sampleSeconds"]):\n        model_config = EsmConfig.from_pretrained(model_path, local_files_only=True)\n        model_config.esmfold_config.fp16_esm = True\n        model = EsmForProteinFolding.from_pretrained(\n            model_path, config=model_config, dtype=torch.float16, device_map={"": "cuda:0"},\n            low_cpu_mem_usage=True, local_files_only=True, weights_only=True,\n        ).eval()\n        # Load directly to the GPU in half precision to bound host RAM; only the language model stays half.\n        for name, child in model.named_children():\n            if name != "esm":\n                child.float()\n        for name, parameter in model.named_parameters(recurse=False):\n            parameter.data = parameter.data.float()\n        model.esm.half()\n        model.trunk.set_chunk_size(prediction["chunkSize"])\n        torch.cuda.synchronize()\n    tokens = torch.tensor([[residue_constants.restype_order[aa] for aa in sequence]], device="cuda")\n    with measurement(root, "prediction", config["measurement"]["sampleSeconds"]) as metrics:\n        torch.cuda.reset_peak_memory_stats()\n        with torch.inference_mode():\n            output = model(tokens, attention_mask=torch.ones_like(tokens), num_recycles=prediction["recycles"])\n        torch.cuda.synchronize()\n        metrics["peakTorchAllocatedBytes"] = torch.cuda.max_memory_allocated()\n        metrics["peakTorchReservedBytes"] = torch.cuda.max_memory_reserved()\n        confidence = output.plddt[0, :, 1].float().cpu().numpy()\n        ptm = float(output.ptm.item())\n        if not np.isfinite(confidence).all() or not np.all((0 <= confidence) & (confidence <= 1)) or not math.isfinite(ptm) or not 0 <= ptm <= 1:\n            raise ValueError("Invalid confidence output")\n        # This pinned Transformers port returns pLDDT in [0,1]; explicitly export PDB B factors in [0,100].\n        pdb_output = dict(output)\n        pdb_output["plddt"] = output.plddt * 100\n        pdb = model.output_to_pdb(pdb_output)[0]\n        structure = parse_pdb(pdb, "A", sequence)\n        (root / "prediction.pdb").write_text(pdb, encoding="ascii")\n        write_json(root / "prediction.json", {\n            "model": prediction["model"], "revision": prediction["revision"],\n            "sequence": sequence, "sequenceSha256": candidate["sequenceSha256"],\n            "plddtRawScale": [0, 1], "plddtExportScale": [0, 100],\n            "meanCaPlddt": float(confidence.mean() * 100), "caPlddt": (confidence * 100).tolist(),\n            "ptm": ptm, "precision": "ESM fp16, folding trunk and heads fp32",\n            "chain": structure["chain"], "residueCount": len(sequence), "atomCount": structure["atomCount"],\n        })\n        np.savez_compressed(root / "predicted-aligned-error.npz",\n                            pae=output.predicted_aligned_error[0].float().cpu().numpy())\n    print("Exact candidate sequence and finite predicted backbone verified", flush=True)\n\n\ndef package(root: Path, config: dict):\n    required = ("environment.json", "reference.json", "reference.pdb", "candidate.json", "prediction.json", "prediction.pdb", "design-summary.json")\n    missing = [file for file in required if not (root / file).is_file()]\n    validations = {"missing": missing, "status": "incomplete"}\n    # Invalidate a previous validation before checking potentially changed inputs.\n    write_json(root / "validation.json", validations)\n    if not missing:\n        reference = parse_pdb((root / "reference.pdb").read_text(), config["reference"]["chain"], config["reference"]["sequence"])\n        draft = prepare_constraints(root, config, reference)\n        if draft:\n            first = parse_fasta((root / "design/seqs/reference.fa").read_text())\n            repeat = parse_fasta((root / "design-repeat/seqs/reference.fa").read_text())\n            if [entry["sequence"] for entry in first] != [entry["sequence"] for entry in repeat] or first[0]["sequence"] != draft["sequence"]:\n                raise ValueError("Constrained design repetition mismatch")\n            if len(first) != config["design"]["samples"] + 1:\n                raise ValueError("Constrained sample count mismatch")\n            verify_constraints(draft, first[1:])\n            if [entry["sequence"] for entry in first[1:]] != [entry["sequence"] for entry in read_json(root / "design-summary.json")["candidates"]]:\n                raise ValueError("Constrained design summary mismatch")\n            if read_json(root / "constraints-validation.json") != constraint_report(draft, config):\n                raise ValueError("Constraint execution report mismatch")\n        candidate = read_json(root / "candidate.json")\n        if draft:\n            verify_constraints(draft, [candidate])\n            if candidate["sequence"] != first[config["design"]["selectedSample"]]["sequence"]:\n                raise ValueError("Constrained sample selection mismatch")\n        recorded_prediction = read_json(root / "prediction.json")\n        sequence_hash = hashlib.sha256(candidate["sequence"].encode("ascii")).hexdigest()\n        if sequence_hash != candidate["sequenceSha256"] or sequence_hash != recorded_prediction["sequenceSha256"] or recorded_prediction["sequence"] != candidate["sequence"]:\n            raise ValueError("Prediction provenance is not associated with the candidate")\n        if not read_json(root / "design-summary.json")["repeatSequencesEqual"]:\n            raise ValueError("Repeated design has not been verified")\n        prediction = parse_pdb((root / "prediction.pdb").read_text(), "A", candidate["sequence"])\n        comparison = compare_ca(reference["ca"], prediction["ca"])\n        write_json(root / "comparison.json", comparison)\n        validations.update(status="completed", exactCandidateSequence=True,\n                           finiteCoordinates=True, completeBackbone=True, repeatedDesign=read_json(root / "design-summary.json")["repeatSequencesEqual"])\n    write_json(root / "validation.json", validations)\n    write_json(root / "result-manifest.json", {\n        "schemaVersion": config["schemaVersion"], "status": validations["status"],\n        "createdAt": datetime.now(timezone.utc).isoformat(), "limitations": config["limitations"],\n        "files": [{"path": str(path.relative_to(root)).replace(os.sep, "/"), "bytes": path.stat().st_size,\n                   "sha256": sha256(path)} for path in sorted(root.rglob("*"))\n                  if path.is_file() and "downloads" not in path.relative_to(root).parts\n                  and path.name != "result-manifest.json"],\n    })\n    archive = root.parent / "proteinforge-design-lab-results.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(root.rglob("*")):\n            if path.is_file() and "downloads" not in path.relative_to(root).parts:\n                bundle.write(path, arcname=path.relative_to(root))\n    # Reopen every exported entry and verify exact bytes against the manifest.\n    with zipfile.ZipFile(archive) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError("ZIP integrity check failed")\n        manifest = json.loads(bundle.read("result-manifest.json"))\n        for entry in manifest["files"]:\n            data = bundle.read(entry["path"])\n            if len(data) != entry["bytes"] or hashlib.sha256(data).hexdigest() != entry["sha256"]:\n                raise ValueError("Reopened result hash mismatch")\n    print(json.dumps({"archive": str(archive), "validation": validations}, indent=2), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("stage", choices=("prepare", "design", "predict", "package"))\n    parser.add_argument("--root", type=Path, required=True)\n    arguments = parser.parse_args()\n    root = arguments.root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    if arguments.stage != "package" and (sys.platform != "linux" or not Path("/content").is_dir()):\n        raise RuntimeError("Run the experiment interactively in the managed Colab runtime")\n    try:\n        config = read_json(root / "config.json")\n        globals()[arguments.stage](root, config)\n    except BaseException as error:\n        write_json(root / "failure.json", {"stage": arguments.stage, "type": type(error).__name__,\n                                           "message": str(error), "traceback": traceback.format_exc()})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
CONFIG_TEXT = '{\n  "schemaVersion": 1,\n  "reference": {\n    "id": "1UBQ",\n    "chain": "A",\n    "url": "https://files.rcsb.org/download/1UBQ.pdb",\n    "sha256": "d4a6812d8951cf6594e6a0763f089e35f5a80b62acb3c117b2c5565228a7b161",\n    "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"\n  },\n  "design": {\n    "repository": "https://github.com/dauparas/ProteinMPNN",\n    "revision": "8907e6671bfbfc92303b5f79c4b5e6ce47cdef57",\n    "rawBase": "https://raw.githubusercontent.com/dauparas/ProteinMPNN/8907e6671bfbfc92303b5f79c4b5e6ce47cdef57/",\n    "files": {\n      "LICENSE": "82009d25ce585631f452b2b24589bdb29c559ccfefa2f200ef312ed5b501a586",\n      "protein_mpnn_run.py": "61f2c519a7f73fa12da9eb90da97b97ec2f8d5f31d42605639c7600cbd321cbe",\n      "protein_mpnn_utils.py": "74c8f9b7553422a7a0bbd705874844ee103c8926c2c96f154a87e0b824071e1b",\n      "vanilla_model_weights/v_48_020.pt": "c9cb4a671d79604111231f8dbfc7c590e06f1197453b7a6854ac6661a642f5bd"\n    },\n    "model": "v_48_020",\n    "seed": 42,\n    "samples": 3,\n    "temperature": 0.1,\n    "selectedSample": 1,\n    "maxLength": 120\n  },\n  "prediction": {\n    "model": "facebook/esmfold_v1",\n    "revision": "75a3841ee059df2bf4d56688166c8fb459ddd97a",\n    "weights": "pytorch_model.bin",\n    "weightsBytes": 8442062570,\n    "weightsSha256": "2ee07356b125d1e3e57503c204111fd7323347fc4735d41d3caac57c2a78e116",\n    "seed": 42,\n    "chunkSize": 32,\n    "recycles": 3,\n    "minimumGpuGiB": 12,\n    "minimumFreeDiskGiB": 20\n  },\n  "measurement": { "sampleSeconds": 0.5 },\n  "download": { "timeoutSeconds": 120, "chunkBytes": 1048576 },\n  "limitations": [\n    "This experiment redesigns a sequence on an experimental backbone; it does not generate a new backbone.",\n    "MPNN likelihood, pLDDT, pTM and backbone RMSD do not demonstrate biological function or experimental stability.",\n    "The selected sample is fixed before prediction; no candidate is selected by predicted confidence.",\n    "Colab runtime versions and hardware are recorded; bitwise reproducibility across different devices is not promised.",\n    "This bundle is an experiment artifact; ProteinForge does not yet import its structures or complete generation history."\n  ]\n}\n'
REQUIREMENTS_TEXT = '# Torch/CUDA are inherited from the managed Colab runtime and recorded in environment.json.\nnumpy==2.2.6\ntransformers==4.57.1\naccelerate==1.11.0\nhuggingface-hub==0.36.0\ntokenizers==0.22.1\nsafetensors==0.6.2\npsutil==7.1.0\n# These optional Transformers imports must remain compatible with NumPy 2.\nlibrosa==0.11.0\n'
CAMPAIGN_TEXT = '"""Run all three prespecified samples in separate, auditable Colab experiments."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef run(root, python):\n    if sys.platform != "linux" or not Path("/content").is_dir():\n        raise RuntimeError("Run this campaign interactively in managed Colab")\n    config = json.loads((root / "config.json").read_text())\n    if config["design"]["samples"] != 3 or config["design"]["selectedSample"] != 1:\n        raise ValueError("The campaign requires three samples and a prespecified first sample")\n    destination = root / "campaign"\n    destination.mkdir()  # Refuse to overwrite a previous or partial campaign.\n    summaries = []\n    for sample in (1, 2, 3):\n        case = destination / f"sample-{sample}" / "run"\n        case.mkdir(parents=True)\n        case_config = json.loads(json.dumps(config))\n        case_config["design"]["selectedSample"] = sample\n        (case / "config.json").write_text(json.dumps(case_config, indent=2) + "\\n")\n        names = ["experiment.py", "constraints.py", "requirements-colab.txt", "installation-report.json"]\n        if config["design"].get("constraints") is not None:\n            names.append("design-draft.json")\n        for name in names:\n            shutil.copyfile(root / name, case / name)\n        for stage in ("prepare", "design", "predict", "package"):\n            print(f"Sample {sample}/3 | {stage}", flush=True)\n            command = [python, "-u", str(case / "experiment.py"), stage, "--root", str(case)]\n            with (case / f"{stage}-stage.log").open("w") as log:\n                with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                      text=True, bufsize=1) as process:\n                    for line in process.stdout:\n                        print(line, end="", flush=True)\n                        log.write(line)\n                    if process.wait():\n                        raise RuntimeError(f"Sample {sample}: {stage} failed; partial files retained at {case}")\n        read = lambda name: json.loads((case / name).read_text())\n        if read("validation.json")["status"] != "completed":\n            raise ValueError(f"Sample {sample} did not complete validation")\n        prediction = read("prediction.json")\n        archive = case.parent / "proteinforge-design-lab-results.zip"\n        summaries.append({\n            "sample": sample, "sequence": prediction["sequence"],\n            "sequenceSha256": prediction["sequenceSha256"],\n            "meanCaPlddt": prediction["meanCaPlddt"], "ptm": prediction["ptm"],\n            "caRmsdAngstrom": read("comparison.json")["caRmsdAngstrom"],\n            "predictionSeconds": read("measurements/prediction.json")["seconds"],\n            "archive": str(archive.relative_to(destination)),\n            "archiveBytes": archive.stat().st_size, "archiveSha256": digest(archive),\n        })\n        print("SAMPLE_RESULT", json.dumps(summaries[-1]), flush=True)\n    # Every case must regenerate the same complete sequence set, not just its selected sample.\n    sequence_sets = [(destination / f"sample-{sample}/run/design/seqs/reference.fa").read_text()\n                     for sample in (1, 2, 3)]\n    sequence_bodies = [[line for line in text.splitlines() if line and not line.startswith(">")]\n                       for text in sequence_sets]\n    if sequence_bodies[0] != sequence_bodies[1] or sequence_bodies[0] != sequence_bodies[2]:\n        raise ValueError("The three independent runs generated different sequence sets")\n    summary = {"status": "completed", "reference": "1UBQ A", "samples": summaries,\n               "sameGeneratedSequenceSet": True, "selection": "All three fixed sample indices; no confidence ranking",\n               "limitations": config["limitations"]}\n    summary_path = destination / "campaign-summary.json"\n    summary_path.write_text(json.dumps(summary, indent=2) + "\\n")\n    combined = root / "proteinforge-design-lab-campaign.zip"\n    with zipfile.ZipFile(combined, "w", compression=zipfile.ZIP_DEFLATED) as bundle:\n        bundle.write(summary_path, "campaign-summary.json")\n        bundle.write(Path(__file__), "campaign.py")\n        for entry in summaries:\n            bundle.write(destination / entry["archive"], entry["archive"])\n    with zipfile.ZipFile(combined) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError("Campaign ZIP failed integrity verification")\n        for entry in summaries:\n            data = bundle.read(entry["archive"])\n            if len(data) != entry["archiveBytes"] or hashlib.sha256(data).hexdigest() != entry["archiveSha256"]:\n                raise ValueError("Campaign ZIP failed sample hash verification")\n    print("CAMPAIGN_RESULT", json.dumps(summary), flush=True)\n    print("CAMPAIGN_ARCHIVE", str(combined), combined.stat().st_size, digest(combined), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--python", default=sys.executable)\n    arguments = parser.parse_args()\n    run(arguments.root.resolve(), arguments.python)\n'
CONSTRAINTS_TEXT = '"""Validate laboratory draft v2 and prepare explicit ProteinMPNN fixed-position input."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\n\nMAX_DRAFT_BYTES = 65_536\nMAX_RESIDUES = 120\nAMINO_ACIDS = re.compile(r"[ACDEFGHIKLMNPQRSTVWY]{3,120}\\Z")\nHASH = re.compile(r"[a-f0-9]{64}\\Z")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef load_draft(path, reference_config, reference, expected_hash):\n    require(path.stat().st_size <= MAX_DRAFT_BYTES, "Draft exceeds the size limit")\n    raw = path.read_bytes()\n    require(HASH.fullmatch(expected_hash) and hashlib.sha256(raw).hexdigest() == expected_hash,\n            "Draft SHA-256 mismatch")\n    draft = json.loads(raw)\n    require(isinstance(draft, dict) and draft.get("kind") == "proteinforge-design-draft"\n            and type(draft.get("schemaVersion")) is int and draft["schemaVersion"] == 2,\n            "Re-export the draft as laboratory draft v2")\n    sequence = draft.get("sequence")\n    require(isinstance(sequence, str) and AMINO_ACIDS.fullmatch(sequence)\n            and len(sequence) == len(reference["sequence"]), "Draft sequence mismatch")\n    source = draft.get("source", {})\n    require(isinstance(source, dict) and source.get("referenceId") == reference_config["id"]\n            and source.get("referenceChain") == reference_config["chain"]\n            and source.get("referenceSequence") == reference["sequence"], "Draft reference mismatch")\n    original = source.get("variantSequence")\n    require(isinstance(original, str) and AMINO_ACIDS.fullmatch(original)\n            and len(original) == len(sequence)\n            and hashlib.sha256(original.encode("ascii")).hexdigest() == source.get("variantSequenceSha256"),\n            "Draft source sequence hash mismatch")\n    require(type(source.get("sample")) is int and 1 <= source["sample"] <= 3, "Draft source sample mismatch")\n    keys = source.get("variantKeys")\n    require(isinstance(keys, list) and len(keys) == len(sequence), "Draft prediction keys mismatch")\n    seen_keys = set()\n    for key in keys:\n        require(isinstance(key, dict) and isinstance(key.get("chain"), str)\n                and re.fullmatch(r"[A-Za-z0-9]", key["chain"])\n                and type(key.get("number")) is int and isinstance(key.get("insertion"), str)\n                and re.fullmatch(r"[A-Za-z]?", key["insertion"]), "Draft prediction key is invalid")\n        identity = (key["chain"], key["number"], key["insertion"])\n        require(identity not in seen_keys, "Duplicate draft prediction key")\n        seen_keys.add(identity)\n    expected_status = "associated-result" if sequence == original else "requires-new-prediction"\n    require(draft.get("constraintStatus") == "not-executed" and draft.get("structureStatus") == expected_status,\n            "Draft execution status mismatch")\n    positions = draft.get("protectedPositions")\n    require(isinstance(positions, list) and 0 < len(positions) < len(sequence),\n            "Protect at least one position and leave at least one position for redesign")\n    seen = set()\n    for entry in positions:\n        require(isinstance(entry, dict) and type(entry.get("chainPosition")) is int,\n                "Invalid protected chain position")\n        position = entry["chainPosition"]\n        require(1 <= position <= len(sequence) and position not in seen, "Duplicate or out-of-range protected position")\n        seen.add(position)\n        number, insertion = reference["residueKeys"][position - 1]\n        require(entry.get("residue") == sequence[position - 1]\n                and entry.get("referenceKey") == {"chain": reference_config["chain"], "number": number, "insertion": insertion}\n                and entry.get("predictionKey") == keys[position - 1], "Protected residue mapping mismatch")\n    return draft\n\n\ndef prepare_constraints(root, config, reference):\n    settings = config.get("design", {}).get("constraints")\n    if settings is None:\n        require(not any((root / name).exists() for name in ("design-draft.json", "design-input.jsonl", "fixed-positions.jsonl", "constraints-validation.json")),\n                "Draft exists without a declared constraint configuration")\n        return None\n    require(isinstance(settings, dict) and set(settings) == {"draftSha256"}, "Invalid constraints configuration")\n    draft = load_draft(root / "design-draft.json", config["reference"], reference, settings["draftSha256"])\n    # JSONL keeps measured backbone atoms and supplies the draft sequence separately.\n    # No experimental PDB residue names or absent side-chain atoms are rewritten.\n    chain = reference["chain"]\n    coordinates = {atom: [] for atom in ("N", "CA", "C", "O")}\n    for line in reference["pdb"].splitlines():\n        atom = line[12:16].strip()\n        if line.startswith("ATOM  ") and atom in coordinates:\n            coordinates[atom].append([float(line[a:b]) for a, b in ((30, 38), (38, 46), (46, 54))])\n    require(all(len(values) == len(draft["sequence"]) for values in coordinates.values()), "Backbone input length mismatch")\n    record = {"name": "reference", "num_of_chains": 1, "seq": draft["sequence"],\n              f"seq_chain_{chain}": draft["sequence"],\n              f"coords_chain_{chain}": {f"{atom}_chain_{chain}": values for atom, values in coordinates.items()}}\n    positions = sorted(entry["chainPosition"] for entry in draft["protectedPositions"])\n    fixed = {"reference": {chain: positions}}\n    for filename, value in (("design-input.jsonl", record), ("fixed-positions.jsonl", fixed)):\n        path = root / filename\n        payload = json.dumps(value, separators=(",", ":")) + "\\n"\n        if path.exists():\n            require(path.read_text() == payload, "Derived constraint input changed")\n        else:\n            path.write_text(payload, encoding="ascii")\n    return draft\n\n\ndef verify_constraints(draft, candidates):\n    for candidate in candidates:\n        sequence = candidate["sequence"]\n        require(len(sequence) == len(draft["sequence"]), "Constrained candidate length mismatch")\n        for entry in draft["protectedPositions"]:\n            require(sequence[entry["chainPosition"] - 1] == entry["residue"],\n                    "Generated sequence violates a protected position")\n\n\ndef constraint_report(draft, config):\n    return {"status": "executed", "draftSha256": config["design"]["constraints"]["draftSha256"],\n            "seedSequence": draft["sequence"], "protectedPositions": draft["protectedPositions"],\n            "samplesCheckedPerRun": config["design"]["samples"], "runsChecked": 2,\n            "allProtectedPositionsPreserved": True}\n'
DEMO_DRAFT_TEXT = '{\n  "kind": "proteinforge-design-draft",\n  "schemaVersion": 2,\n  "sequence": "MRIKVEFEDGTTLTLEVSPEDTIAELKAKIQEKTGIPPEEQRLIYKGKVLEDDKTLADYGIKEGDTIRLELVPKGG",\n  "source": {\n    "referenceId": "1UBQ",\n    "referenceChain": "A",\n    "referenceSequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",\n    "variantSequence": "MTIKVEFEDGTTLTLEVSPEDTIAELKAKIQEKTGIPPEEQRLIYKGKVLEDDKTLADYGIKEGDTIRLELVPKGG",\n    "variantKeys": [\n      {\n        "chain": "A",\n        "number": 1,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 2,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 3,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 4,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 5,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 6,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 7,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 8,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 9,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 10,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 11,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 12,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 13,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 14,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 15,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 16,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 17,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 18,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 19,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 20,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 21,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 22,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 23,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 24,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 25,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 26,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 27,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 28,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 29,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 30,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 31,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 32,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 33,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 34,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 35,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 36,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 37,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 38,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 39,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 40,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 41,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 42,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 43,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 44,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 45,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 46,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 47,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 48,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 49,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 50,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 51,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 52,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 53,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 54,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 55,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 56,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 57,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 58,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 59,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 60,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 61,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 62,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 63,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 64,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 65,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 66,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 67,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 68,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 69,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 70,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 71,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 72,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 73,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 74,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 75,\n        "insertion": ""\n      },\n      {\n        "chain": "A",\n        "number": 76,\n        "insertion": ""\n      }\n    ],\n    "variantSequenceSha256": "39e5d092045ebb891a46d3e2c09c6323faddedc5f2a445e9bd9a67da0bfc2b6d",\n    "sample": 1\n  },\n  "structureStatus": "requires-new-prediction",\n  "constraintStatus": "not-executed",\n  "protectedPositions": [\n    {\n      "chainPosition": 2,\n      "residue": "R",\n      "referenceKey": {\n        "chain": "A",\n        "number": 2,\n        "insertion": ""\n      },\n      "predictionKey": {\n        "chain": "A",\n        "number": 2,\n        "insertion": ""\n      }\n    },\n    {\n      "chainPosition": 7,\n      "residue": "F",\n      "referenceKey": {\n        "chain": "A",\n        "number": 7,\n        "insertion": ""\n      },\n      "predictionKey": {\n        "chain": "A",\n        "number": 7,\n        "insertion": ""\n      }\n    },\n    {\n      "chainPosition": 42,\n      "residue": "R",\n      "referenceKey": {\n        "chain": "A",\n        "number": 42,\n        "insertion": ""\n      },\n      "predictionKey": {\n        "chain": "A",\n        "number": 42,\n        "insertion": ""\n      }\n    }\n  ]\n}'
#@title Preparar el entorno de Colab / Prepare the Colab environment
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import datetime, timezone

if sys.platform != "linux" or importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Open this notebook in a managed Google Colab GPU runtime")
CONFIG = json.loads(CONFIG_TEXT)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     check=True, capture_output=True, text=True).stdout.strip()
print("Allocated GPU:", gpu)
if float(gpu.splitlines()[0].rsplit(",", 1)[1]) / 1024 < CONFIG["prediction"]["minimumGpuGiB"]:
    raise RuntimeError("This notebook needs at least 12 GiB VRAM; do not switch to paid resources")
WORK = Path("/content/proteinforge-design-lab")
WORK.mkdir(exist_ok=True)
if shutil.disk_usage(WORK).free < CONFIG["prediction"]["minimumFreeDiskGiB"] * 1024 ** 3:
    raise RuntimeError("Not enough free space in the managed runtime")
RUN = WORK / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S%fZ")
RUN.mkdir()
(RUN / "config.json").write_text(CONFIG_TEXT, encoding="utf-8")
(RUN / "experiment.py").write_text(PROGRAM_TEXT, encoding="utf-8")
(RUN / "requirements-colab.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")
(RUN / "campaign.py").write_text(CAMPAIGN_TEXT, encoding="utf-8")
(RUN / "constraints.py").write_text(CONSTRAINTS_TEXT, encoding="utf-8")
ENV = WORK / "venv"
# Managed Debian runtimes may omit ensurepip; inherit their existing pip instead.
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(ENV)], check=True)
PYTHON = str(ENV / "bin" / "python")
subprocess.run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(RUN / "installation-report.json"),
                "-r", str(RUN / "requirements-colab.txt")], check=True)
STAGE_ENV = dict(os.environ, HF_HOME=str(WORK / "model-cache"),
                 HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1", USE_TF="0", USE_FLAX="0")

def run_stage(name):
    command = [PYTHON, "-u", str(RUN / "experiment.py"), name, "--root", str(RUN)]
    print("Stage:", name, "| Results:", RUN, flush=True)
    with (RUN / f"{name}-stage.log").open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, env=STAGE_ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
    if returncode:
        failure = RUN / "failure.json"
        if failure.exists():
            print(failure.read_text(), flush=True)
        raise RuntimeError(f"Stage {name} failed. Read its output; the final download cell can export partial evidence.")
print("Environment ready. No private sequence or Drive mount is used.")

## Posiciones protegidas opcionales / Optional fixed positions

Antes de preparar la referencia, elige `none` (ensayo original), `public-demo` (borrador público: R2, F7, R42) o `upload` (archivo JSON v2 descargado del laboratorio). La demo cambia T2 a R y lo protege: no debe copiar Q2 de la referencia experimental. / Before preparing the reference, choose none, public-demo (R2/F7/R42), or upload a laboratory draft v2. The demo changes T2 to R and fixes it; the experimental reference's Q2 must not replace it.

`upload` envía el archivo que elijas a este entorno de Google Colab. Usa solo datos que tengas autorización para procesar allí. No se montará Drive. Máximo 64 KiB, misma referencia/longitud y al menos una posición libre. Solo las posiciones protegidas quedan fijadas; las demás pueden cambiar. / Upload sends the selected file to this Google Colab runtime. Use only authorized data; no Drive mount. Maximum 64 KiB, matching reference/length and at least one free position. Only protected positions are fixed; other residues may change.

La generación verifica todas las muestras y la repetición; empaquetar vuelve a verificar las posiciones. Un fallo no se presenta como ejecución restringida correcta. / Every sample and repeated run is checked, then checked again before packaging. Failed constraints cannot be reported as successful.

In [ ]:
DRAFT_MODE = "none"  # none | public-demo | upload
if DRAFT_MODE not in ("none", "public-demo", "upload"):
    raise ValueError("Unknown draft mode")
if DRAFT_MODE != "none":
    import hashlib
    if DRAFT_MODE == "public-demo":
        draft_bytes = DEMO_DRAFT_TEXT.encode("utf-8")
    else:
        from google.colab import files
        uploaded_drafts = files.upload()
        if len(uploaded_drafts) != 1:
            raise ValueError("Select exactly one laboratory draft JSON file")
        draft_bytes = next(iter(uploaded_drafts.values()))
    if len(draft_bytes) > 65536:
        raise ValueError("Draft exceeds 64 KiB")
    draft_path = RUN / "design-draft.json"
    if draft_path.exists() or (RUN / "provenance.json").exists():
        raise RuntimeError("Start a fresh setup before changing constraints; previous evidence is preserved")
    draft_path.write_bytes(draft_bytes)
    CONFIG["design"]["constraints"] = {"draftSha256": hashlib.sha256(draft_bytes).hexdigest()}
    (RUN / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n", encoding="utf-8")
    print("Draft recorded. Protected positions will be validated before generation.")

## Referencia e integridad / Reference and integrity
Descarga solo datos públicos y archivos oficiales con revisión y SHA-256 fijados. Valida la cadena y todos los átomos del esqueleto. / Download public data and pinned official files; validate the chain and complete backbone atoms.

In [ ]:
run_stage("prepare")

## Rediseño / Sequence redesign
Genera tres secuencias y repite con la misma semilla. La muestra 1 se selecciona **antes de predecir**, sin elegirla por su confianza posterior. El score de ProteinMPNN es una puntuación del modelo, no energía física. / Generate three sequences, repeat with the same seed, and select sample 1 before prediction. ProteinMPNN scores are model likelihood scores, not physical energy.

In [ ]:
run_stage("design")
print((RUN / "candidate.fasta").read_text())

## Predicción / Prediction
ESMFold recibe la secuencia exacta seleccionada. Valida coordenadas finitas, residuos y esqueleto completo. Registra pLDDT, pTM, memoria y tiempo. Este port entrega pLDDT en 0–1; el PDB y el resumen se exportan explícitamente en 0–100. No es probabilidad de función. / ESMFold receives the exact selected sequence; coordinates, residues and backbone completeness are validated. Confidence, memory and time are recorded; this port's 0–1 pLDDT is explicitly exported on a 0–100 scale. It is not a probability of function.

In [ ]:
run_stage("predict")
prediction_summary = json.loads((RUN / "prediction.json").read_text())
print(json.dumps({key: prediction_summary[key] for key in
    ("model", "residueCount", "meanCaPlddt", "ptm", "precision")}, indent=2))

## Comparar y descargar / Compare and download
RMSD de Cα tras superposición rígida sin reflexión, sin excluir residuos. No comparar como si ambas secuencias fueran idénticas: cada posición corresponde al esqueleto de referencia usado para diseñar. El ZIP excluye pesos/cachés y se reabre para verificar sus hashes. / Cα RMSD after rigid superposition without reflection or residue exclusion. Positional correspondence comes from the design backbone, not from identical sequences. Weights and caches are excluded; the ZIP is reopened and its hashes checked.

Si una etapa falla, ejecuta esta última celda para descargar evidencia parcial: figurará **incomplete**, sin inventar resultados. Abre el ZIP completo en Diseñar → Laboratorio para validarlo e inspeccionarlo. / If a stage fails, run this final cell to export partial evidence marked incomplete. Open a completed ZIP in Design → Laboratory for validation and inspection.

In [ ]:
run_stage("package")
if (RUN / "comparison.json").exists():
    comparison_summary = json.loads((RUN / "comparison.json").read_text())
    print(json.dumps({key: comparison_summary[key] for key in
        ("method", "residuesCompared", "caRmsdAngstrom")}, indent=2))
print((RUN / "validation.json").read_text())
from google.colab import files
files.download(str(WORK / "proteinforge-design-lab-results.zip"))

## Tres variantes opcionales / Optional three-variant campaign

Para comprobar todas las muestras, cambia `RUN_CAMPAIGN` a `True` y ejecuta esta celda después de preparar el entorno. Cada muestra repite las cuatro etapas en su propia carpeta; la primera también se ejecuta desde cero con la fuente corregida. No se elige una ganadora por confianza ni se generan tres formas independientes. La caché de pesos se comparte, los resultados no. / To verify every sample, set `RUN_CAMPAIGN` to `True` after environment setup. Each sample independently repeats all four stages, including a clean first sample. There is no confidence-based winner or three independent backbone designs. Only the model cache is shared.

Una carpeta existente se conserva y se rechaza sobrescribirla. Si Colab pierde el entorno, vuelve a ejecutar la preparación. / Existing campaign folders are preserved and cannot be overwritten. Repeat setup if Colab loses its runtime.

In [ ]:
RUN_CAMPAIGN = False
if RUN_CAMPAIGN:
    with subprocess.Popen([PYTHON, "-u", str(RUN / "campaign.py"), "--root", str(RUN)],
                          env=STAGE_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError("Campaign failed; inspect the stage output and retained partial files")
    print((RUN / "campaign/campaign-summary.json").read_text())
else:
    print("Campaña opcional desactivada / Optional campaign disabled")